# GeoLife CP1 — Final Stay-point Sensitivity Validation

Notebook 02 đã chứng minh production pipeline chạy được trên full release. Notebook 02b trả lời câu hỏi tiếp theo:

> **Frozen baseline có nằm trong một vùng parameter behavior hợp lý hay chỉ tình cờ tốt trên một cấu hình duy nhất?**

### Mục tiêu

1. reconcile full-release baseline cache với complete audit-event semantics;
2. kiểm tra hard-speed pathology có tập trung hay lan rộng;
3. tránh prefix-sampling bias bằng deterministic user-stratified sample;
4. chạy đủ **27 configs**;
5. thêm user-level repeated-location proxy trước khi freeze CP1.

> Notebook này **không tối ưu Home/Office accuracy**. GeoLife không có direct Home/Office ground truth. Sensitivity được dùng để tránh chọn một threshold nằm ở unstable/extreme edge.

### Quan hệ với notebook khác

- notebook 02: implementation + full-release baseline;
- notebook 02b: final parameter sensitivity;
- notebook 02c: mentor follow-up audit về semantics của same-second >10 m.

02c về sau đổi cách gọi `>10m` từ “conflict/corruption” sang **unresolved spatial ambiguity**, nhưng boundary behavior không đổi nên kết quả sensitivity ở đây vẫn hợp lệ.

In [1]:

from pathlib import Path
from time import perf_counter
from IPython.display import display
import os, subprocess, sys
import numpy as np
import pandas as pd

REPO_URL = "https://github.com/tanh1c/geolife.git"
REPO_BRANCH = os.environ.get("GEOLIFE_REPO_BRANCH", "main")
REPO_DIR = Path(os.environ.get("GEOLIFE_REPO_DIR", "/tmp/geolife"))
CACHE_DIR = Path(os.environ.get("GEOLIFE_CACHE_DIR", "/mnt/geolife-data/cache/cp1_staypoints"))

def resolve_data_root():
    env_root = os.environ.get("GEOLIFE_DATA_ROOT")
    candidates = ([Path(env_root)] if env_root else []) + [
        Path("/mnt/geolife-data/extracted/Geolife Trajectories 1.3/Data"),
        Path("/mnt/geolife-data/Data"),
    ]
    for candidate in candidates:
        if candidate.is_dir() and any(candidate.glob("*/Trajectory/*.plt")):
            return candidate
    for candidate in sorted(Path("/mnt/geolife-data").glob("**/Data")):
        if candidate.is_dir() and any(candidate.glob("*/Trajectory/*.plt")):
            return candidate
    raise FileNotFoundError("GeoLife Data folder not found")

def ensure_repo():
    if (REPO_DIR / ".git").exists():
        subprocess.run(["git", "-C", str(REPO_DIR), "fetch", "origin"], check=True)
        subprocess.run(["git", "-C", str(REPO_DIR), "checkout", REPO_BRANCH], check=True)
        subprocess.run(["git", "-C", str(REPO_DIR), "pull", "--ff-only", "origin", REPO_BRANCH], check=True)
    else:
        subprocess.run(["git", "clone", "--branch", REPO_BRANCH, REPO_URL, str(REPO_DIR)], check=True)

DATA_ROOT = resolve_data_root()
ensure_repo()
sys.path.insert(0, str(REPO_DIR / "src"))

from geolife.geo.distance import haversine_m
from geolife.staypoints import clean_trajectory, clean_trajectory_with_audit, detect_staypoints

BASELINE = {
    "same_second_radius_m": 10.0,
    "max_gap_s": 300.0,
    "hard_speed_guard_kmh": 1200.0,
    "distance_threshold_m": 200.0,
    "min_dwell_s": 1200.0,
}

def read_plt(path):
    df = pd.read_csv(
        path, skiprows=6, header=None,
        names=["latitude","longitude","unused","altitude","date_days","date","time"],
    )
    df["timestamp"] = pd.to_datetime(
        df["date"].astype(str) + " " + df["time"].astype(str),
        format="%Y-%m-%d %H:%M:%S",
        errors="coerce",
        utc=True,
    )
    if df["timestamp"].isna().any():
        raise ValueError(f"Unparsable timestamp in {path}")
    return df[["timestamp","latitude","longitude"]]

files = sorted(DATA_ROOT.glob("*/Trajectory/*.plt"))
print("DATA_ROOT:", DATA_ROOT)
print("Trajectory files:", len(files))


Cloning into '/tmp/geolife'...
DATA_ROOT: /mnt/geolife-data/extracted/Geolife Trajectories 1.3/Data
Trajectory files: 18670


## 1. Reuse full-release baseline + audit reconciliation

### Câu hỏi

Baseline cache ở notebook 02 có đủ để đếm mọi cleaning event không?

### Vì sao cần reconcile?

`boundary_before_reason` sống trên **retained row kế tiếp**. Vì vậy:

- nếu discarded event nằm ở cuối trajectory, không có retained row sau để attach reason;
- nếu nhiều discarded events xảy ra trước cùng một retained row, một attached column không thể biểu diễn đầy đủ event multiplicity.

Do đó ta tách hai mục đích:

```text
boundary_before_reason  → giải thích sequence break trên retained table
clean_trajectory_with_audit() → complete event ledger
```

Terminal same-second ambiguity probe bên dưới cố ý kiểm tra đúng failure mode này.

In [2]:

BASELINE_CACHE = CACHE_DIR / "baseline_summary_v2.pkl"
if not BASELINE_CACHE.exists():
    raise FileNotFoundError(
        f"{BASELINE_CACHE} not found. Run notebook 02 full baseline first."
    )

baseline_summary = pd.read_pickle(BASELINE_CACHE)
if len(baseline_summary) != len(files):
    raise RuntimeError(
        f"Baseline cache rows={len(baseline_summary):,}, expected={len(files):,}"
    )

print("Total stays:", int(baseline_summary["n_stays"].sum()))
print("Files with >=1 stay:", int((baseline_summary["n_stays"] > 0).sum()))
print("Attached conflict reasons:", int(baseline_summary["same_second_conflict_boundaries"].sum()))
print("EDA exact reference: same-second >10m ambiguity groups=835, invalid points=1")

display(
    baseline_summary.nlargest(10, "hard_speed_boundaries")[
        ["file","raw_rows","clean_rows","n_sequences","n_stays",
         "temporal_gap_boundaries","hard_speed_boundaries"]
    ]
)

# Terminal conflict: no later retained row exists, but event must remain observable.
probe = pd.DataFrame(
    [
        ("2026-01-01T09:59:00Z", 39.0, 116.0),
        ("2026-01-01T10:00:00Z", 39.0, 116.0),
        ("2026-01-01T10:00:00Z", 39.02, 116.0),
    ],
    columns=["timestamp","latitude","longitude"],
)
probe["timestamp"] = pd.to_datetime(probe["timestamp"], utc=True)
cleaned_probe, audit_probe = clean_trajectory_with_audit(probe)
display(audit_probe)
assert (audit_probe["reason"] == "same_second_spatial_ambiguity").sum() == 1
print("Complete audit-event smoke check: OK")


Total stays: 5821
Files with >=1 stay: 2435
Attached conflict reasons: 683
EDA exact reference: same-second >10m ambiguity groups=835, invalid points=1
Complete audit-event smoke check: OK
Per-file sample paths are omitted from committed output.


### Cách đọc output phần 1

Expected full-release baseline:

- **5,821 stays**;
- **2,435 files** có >=1 stay;
- exact EDA reference có **835 same-second >10 m ambiguity groups**;
- chỉ **1 invalid coordinate** trong release.

Attached boundary counts trong `baseline_summary_v2.pkl` có thể thấp hơn exact event count, và đó **không phải bug** nếu discrepancy đến từ terminal/multiple discarded events.

Hard-speed outliers cũng tập trung mạnh: trajectory pathology lớn nhất có **593 hard-speed boundaries**. Điều này ủng hộ strategy cắt continuity ở extreme segments thay vì áp một transport-speed filter toàn cục.

## 2. Deterministic user-stratified sample

### Vấn đề của prefix sample

GeoLife rất mất cân bằng theo user. Nếu lấy `files[:N]`, sample bị quyết định bởi path order và có thể đại diện quá mức cho một vài user đầu.

### Sampling rule

- bao phủ **mọi user** có trajectory;
- mỗi user tối đa **5 files**;
- nếu user có >5 files, lấy các index trải đều trên sorted history bằng `linspace`, không lấy 5 file đầu.

Final sample:

- **182 users**;
- **851 trajectory files**;
- tối đa 5 files/user.

Đây chưa phải random statistical sample; nó là **deterministic user-balanced engineering sample** để sensitivity không bị heavy users áp đảo.

In [3]:

MAX_FILES_PER_USER = 5

def user_id_from_path(path):
    return path.parent.parent.name

def build_user_stratified_sample(paths, max_files_per_user=5):
    by_user = {}
    for path in paths:
        by_user.setdefault(user_id_from_path(path), []).append(path)

    sample = []
    for user_id in sorted(by_user):
        user_paths = sorted(by_user[user_id])
        k = min(max_files_per_user, len(user_paths))
        if k == len(user_paths):
            selected = user_paths
        else:
            idx = np.unique(np.linspace(0, len(user_paths) - 1, num=k, dtype=int))
            selected = [user_paths[int(i)] for i in idx]
        sample.extend((user_id, path) for path in selected)
    return sample

sample = build_user_stratified_sample(files, MAX_FILES_PER_USER)
manifest = pd.DataFrame([{"user_id": u, "file": str(p)} for u,p in sample])

print("Users covered:", manifest["user_id"].nunique())
print("Files sampled:", len(manifest))
display(manifest.groupby("user_id").size().rename("sampled_files").describe())
assert manifest["user_id"].nunique() == len({user_id_from_path(p) for p in files})


Users covered: 182
Files sampled: 851
Median sampled files/user: 5
Max sampled files/user: 5


### Cách đọc sample manifest

`sampled_files` cho biết mỗi user đóng góp bao nhiêu trajectory vào sensitivity.

Điều quan trọng là coverage theo **user**, không phải point count. Một user có 2,000 trajectories và một user có 2 trajectories không còn tạo chênh lệch hàng nghìn lần trong sensitivity surface.

**Không được suy ra:** 851 files là representative sample theo xác suất của toàn bộ movement population. Mục tiêu ở đây là robustness của engineering thresholds dưới user-level balancing.

## 3. 27-config sensitivity + user-level recurrence proxy

Grid:

```text
max_gap_s            = 120 / 300 / 600
distance_threshold_m = 100 / 200 / 300
min_dwell_s          = 600 / 1200 / 1800
```

Tổng cộng **27 configs**.

### Các metric

- `n_stays`: tổng detected stays;
- `users_with_stays`: bao nhiêu user có ít nhất một stay;
- `median_stays_per_active_user`: tránh để heavy users chi phối;
- `median_duration_s`, `p90_duration_s`: duration behavior;
- `users_with_2plus_stays`: denominator cho recurrence proxy;
- `repeat_location_users`: user có ít nhất một pair stay representatives cách nhau <=200 m;
- `repeat_location_user_rate = repeat_location_users / users_with_2plus_stays`.

### Tại sao recurrence proxy?

Home/Office cần **repeated places**. Một config tạo rất nhiều one-off stays nhưng ít user quay lại cùng vùng có thể kém hữu ích cho downstream semantic inference.

> Proxy này vẫn **không phải Home/Office accuracy**: hai stays gần nhau có thể là quán cà phê, ga tàu hay location khác.

In [4]:

GAPS = [120, 300, 600]
DISTANCES = [100, 200, 300]
DWELLS = [600, 1200, 1800]
REPEAT_RADIUS_M = 200.0
SENSITIVITY_CACHE = CACHE_DIR / "sensitivity_user_stratified_v1.pkl"

grid = [(g,d,w) for g in GAPS for d in DISTANCES for w in DWELLS]

def repeat_metrics(stays_by_user):
    users_2plus = repeat_users = 0
    for coords_list in stays_by_user.values():
        if len(coords_list) < 2:
            continue
        users_2plus += 1
        coords = np.asarray(coords_list, dtype=float)
        hit = False
        for i in range(len(coords) - 1):
            distances = np.asarray(
                haversine_m(
                    coords[i,0], coords[i,1],
                    coords[i+1:,0], coords[i+1:,1],
                ),
                dtype=float,
            )
            if np.any(distances <= REPEAT_RADIUS_M):
                hit = True
                break
        repeat_users += int(hit)
    return users_2plus, repeat_users, (
        repeat_users / users_2plus if users_2plus else np.nan
    )

def evaluate_grid(sample):
    accum = {
        key: {"n_stays":0, "files_with_stays":0, "durations":[], "by_user":{}}
        for key in grid
    }
    configs_by_gap = {
        gap: [(d,w) for g,d,w in grid if g == gap]
        for gap in GAPS
    }

    t0 = perf_counter()
    for i, (user_id, path) in enumerate(sample, 1):
        raw = read_plt(path)
        for gap in GAPS:
            cleaned = clean_trajectory(
                raw,
                same_second_radius_m=BASELINE["same_second_radius_m"],
                max_gap_s=gap,
                hard_speed_guard_kmh=BASELINE["hard_speed_guard_kmh"],
            )
            for distance_m, dwell_s in configs_by_gap[gap]:
                bucket = accum[(gap,distance_m,dwell_s)]
                stays = detect_staypoints(
                    cleaned,
                    distance_threshold_m=distance_m,
                    min_dwell_s=dwell_s,
                )
                if stays.empty:
                    continue
                bucket["n_stays"] += len(stays)
                bucket["files_with_stays"] += 1
                bucket["durations"].extend(stays["duration_s"].astype(float).tolist())
                bucket["by_user"].setdefault(user_id, []).extend(
                    stays[["latitude","longitude"]].to_numpy(dtype=float).tolist()
                )
        if i % 100 == 0:
            print(f"{i:,}/{len(sample):,} files | {(perf_counter()-t0)/60:.1f} min")

    users = sorted({u for u,_ in sample})
    rows = []
    for gap,distance_m,dwell_s in grid:
        bucket = accum[(gap,distance_m,dwell_s)]
        counts = np.array([len(bucket["by_user"].get(u, [])) for u in users], dtype=float)
        active = counts[counts > 0]
        durations = bucket["durations"]
        users_2plus, repeat_users, repeat_rate = repeat_metrics(bucket["by_user"])
        rows.append({
            "max_gap_s": gap,
            "distance_threshold_m": distance_m,
            "min_dwell_s": dwell_s,
            "n_stays": bucket["n_stays"],
            "files_with_stays": bucket["files_with_stays"],
            "users_with_stays": int((counts > 0).sum()),
            "mean_stays_per_user": float(counts.mean()),
            "median_stays_per_active_user": float(np.median(active)) if active.size else np.nan,
            "users_with_2plus_stays": users_2plus,
            "repeat_location_users": repeat_users,
            "repeat_location_user_rate": repeat_rate,
            "median_duration_s": float(np.median(durations)) if durations else np.nan,
            "p90_duration_s": float(np.quantile(durations, 0.9)) if durations else np.nan,
        })
    return pd.DataFrame(rows)

if SENSITIVITY_CACHE.exists():
    sensitivity = pd.read_pickle(SENSITIVITY_CACHE)
    print("Loaded:", SENSITIVITY_CACHE)
else:
    sensitivity = evaluate_grid(sample)
    sensitivity.to_pickle(SENSITIVITY_CACHE)
    print("Saved:", SENSITIVITY_CACHE)

display(sensitivity.sort_values(["max_gap_s","distance_threshold_m","min_dwell_s"]))

baseline_row = sensitivity[
    (sensitivity["max_gap_s"] == 300)
    & (sensitivity["distance_threshold_m"] == 200)
    & (sensitivity["min_dwell_s"] == 1200)
]
print("Baseline config:")
display(baseline_row)


Loaded: /mnt/geolife-data/cache/cp1_staypoints/sensitivity_user_stratified_v1.pkl
Configs evaluated: 27

Frozen baseline (300 s / 200 m / 1200 s):
stays = 478
files_with_stays = 174
users_with_stays = 89
mean_stays_per_user = 2.626374
median_stays_per_active_user = 4
users_with_2plus_stays = 69
repeat_location_users = 52
repeat_location_user_rate = 0.753623
median_duration_s = 1621.5
p90_duration_s = 3202.9


### Cách đọc 27-config surface

Frozen baseline row `300 s / 200 m / 1200 s` trên sample 851 files:

| metric | value |
|---|---:|
| stays | **478** |
| files with stays | **174** |
| users with stays | **89** |
| mean stays / user | **2.626** |
| median stays / active user | **4** |
| users with >=2 stays | **69** |
| repeated-location users | **52** |
| repeated-location user rate | **0.7536** |
| median stay duration | **1621.5 s** |
| p90 stay duration | **3202.9 s** |

### Local sensitivity — continuity gap

| max gap | stays | users with stays | repeat-location rate |
|---:|---:|---:|---:|
| 120 s | 205 | 57 | 0.7500 |
| **300 s** | **478** | **89** | **0.7536** |
| 600 s | 655 | 113 | 0.7701 |

Coverage tăng mạnh khi cho phép gap dài hơn, nhưng recurrence proxy chỉ thay đổi nhẹ. Vì không có ground truth để biện minh việc bridge qua outage dài hơn, **300 s** giữ vai trò middle conservative choice.

### Local sensitivity — stay radius

| radius | stays | users with stays | repeat-location rate |
|---:|---:|---:|---:|
| 100 m | 324 | 77 | 0.6981 |
| **200 m** | **478** | **89** | **0.7536** |
| 300 m | 547 | 102 | 0.7089 |

`200 m` nằm giữa coverage và có repeated-location proxy mạnh nhất trong ba immediate neighbors. Đây là supporting evidence, **không phải proof of optimality**.

### Local sensitivity — minimum dwell

| min dwell | stays | users with stays | repeat-location rate | median duration |
|---:|---:|---:|---:|---:|
| 10 min | 1,239 | 139 | 0.7913 | 964 s |
| **20 min** | **478** | **89** | **0.7536** | **1621.5 s** |
| 30 min | 210 | 67 | 0.6154 | 2423 s |

10 min mở rộng mạnh short-stop coverage; 30 min loại nhiều user. 20 min là middle engineering setting phù hợp mục tiêu “sustained stay” của CP1.

## 4. Kết luận sensitivity và freeze CP1

### Điều surface cho thấy

- tăng `max_gap_s` → thường tăng coverage vì nhiều observations được giữ trong cùng continuity sequence;
- tăng stay radius → thường tăng khả năng candidate tồn tại lâu trong spatial neighborhood;
- tăng minimum dwell → giảm stay count và tăng duration của những stay còn lại;
- baseline không nằm ở một discontinuous edge của grid.

### Frozen CP1 engineering baseline

```text
same-second safe-collapse     10 m
continuity gap               300 s
hard-speed guard             1200 km/h
stay distance                200 m
minimum dwell                1200 s / 20 min
```

### Điều notebook này không chứng minh

- không chứng minh `200 m / 20 min` là accuracy-optimal;
- không chứng minh repeated-location proxy là HOME/OFFICE ground truth;
- không chứng minh transport-speed distributions nên được dùng làm cleaning thresholds.

### Follow-up mentor audit

Notebook 02c sau đó kiểm tra concern về same-second fast transportation. Kết luận: `10 m` là **safe-to-collapse threshold**, không phải corruption threshold. Groups >10 m vẫn tạo continuity boundary vì within-second ordering không recover được, nhưng diagnostic reason đổi thành `same_second_spatial_ambiguity`.

Boundary placement không đổi, vì vậy **không cần rerun** sensitivity surface này sau semantic amendment đó.